# Traffic Demand Prediction — Improved v2
**Score = max(0, 100 × R²)**  
Improvements over v1:
- Fix timestamp parsing (H:M string split instead of pd.to_datetime)
- Keep `day` as a feature
- Cyclical encoding (sin/cos) for hour & minute
- Target encoding for high-cardinality `geohash`
- LightGBM + XGBoost + RandomForest ensemble
- Optuna hyperparameter tuning
- Better missing value imputation
- Feature interactions

In [1]:
# ── 1. Imports ────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import KFold, cross_val_score
from sklearn import metrics
import lightgbm as lgb
import xgboost as xgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
import warnings
warnings.filterwarnings('ignore')
print('All imports OK')

All imports OK


In [2]:
# ── 2. Load Data ──────────────────────────────────────────────────────────────
train = pd.read_csv('train.csv')
test  = pd.read_csv('test.csv')

print('Train shape:', train.shape)
print('Test shape :', test.shape)
print('\nTrain head:')
display(train.head())
print('\nMissing values (train):')
print(train.isnull().sum())
print('\nMissing values (test):')
print(test.isnull().sum())
print('\nDemand stats:')
print(train['demand'].describe())

Train shape: (77299, 11)
Test shape : (41778, 10)

Train head:


,Index,geohash,day,timestamp,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather
0,0,qp02z1,48,0:0,0.048804,NaN,1,Not Allowed,No,NaN,NaN
1,1,qp02zt,48,0:0,0.118507,Residential,3,Allowed,Yes,31.104565,Sunny
2,2,qp08bj,48,0:0,0.027132,Residential,1,Not Allowed,No,25.919267,Sunny
3,3,qp08gt,48,0:0,0.003272,Residential,1,Not Allowed,No,NaN,Rainy
4,4,qp02zq,48,0:0,0.010819,Residential,1,Not Allowed,No,10.803667,Rainy



Missing values (train):
Index               0
geohash             0
day                 0
timestamp           0
demand              0
RoadType          600
NumberofLanes       0
LargeVehicles       0
Landmarks           0
Temperature      2495
Weather           797
dtype: int64

Missing values (test):
Index               0
geohash             0
day                 0
timestamp           0
RoadType          324
NumberofLanes       0
LargeVehicles       0
Landmarks           0
Temperature      1349
Weather           431
dtype: int64

Demand stats:
count    7.729900e+04
mean     9.394238e-02
std      1.421905e-01
min      6.245650e-07
25%      1.822723e-02
50%      4.775994e-02
75%      1.085951e-01
max      1.000000e+00
Name: demand, dtype: float64


In [3]:
# ── 3. Feature Engineering ────────────────────────────────────────────────────
def engineer_features(df):
    df = df.copy()

    # --- Fix timestamp parsing: format is "H:M" ---
    ts_parts = df['timestamp'].astype(str).str.split(':', expand=True)
    df['hour']   = ts_parts[0].astype(int)
    df['minute'] = ts_parts[1].astype(int)

    # --- Total minutes since midnight (continuous time feature) ---
    df['time_minutes'] = df['hour'] * 60 + df['minute']

    # --- Cyclical encoding for hour and minute ---
    df['hour_sin']   = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos']   = np.cos(2 * np.pi * df['hour'] / 24)
    df['minute_sin'] = np.sin(2 * np.pi * df['minute'] / 60)
    df['minute_cos'] = np.cos(2 * np.pi * df['minute'] / 60)
    df['time_sin']   = np.sin(2 * np.pi * df['time_minutes'] / 1440)
    df['time_cos']   = np.cos(2 * np.pi * df['time_minutes'] / 1440)

    # --- Rush-hour flags (more granular) ---
    df['morning_rush']  = ((df['hour'] >= 7) & (df['hour'] <= 10)).astype(int)
    df['evening_rush']  = ((df['hour'] >= 17) & (df['hour'] <= 20)).astype(int)
    df['night']         = ((df['hour'] >= 22) | (df['hour'] <= 5)).astype(int)
    df['rush_hour']     = ((df['morning_rush'] == 1) | (df['evening_rush'] == 1)).astype(int)

    # --- Time period buckets ---
    df['time_period'] = pd.cut(df['hour'],
                               bins=[-1, 5, 10, 15, 20, 24],
                               labels=['night', 'morning', 'midday', 'evening', 'late_night'])

    # --- Keep day as feature (it's numeric!) ---
    # day is already numeric (48, 49, etc.)

    # --- Geohash features ---
    df['geo_prefix']  = df['geohash'].astype(str).str[:4]
    df['geo_prefix3'] = df['geohash'].astype(str).str[:3]
    df['geo_len']     = df['geohash'].astype(str).str.len()

    # --- Temperature missing flag (missingness can be informative) ---
    df['temp_missing'] = df['Temperature'].isnull().astype(int)

    # --- RoadType missing flag ---
    df['road_missing'] = df['RoadType'].isnull().astype(int)

    return df

train = engineer_features(train)
test  = engineer_features(test)
print('After feature engineering:', train.shape)
print('New columns:', [c for c in train.columns if c not in ['Index','geohash','day','timestamp','demand','RoadType','NumberofLanes','LargeVehicles','Landmarks','Temperature','Weather']])

After feature engineering: (77299, 30)
New columns: ['hour', 'minute', 'time_minutes', 'hour_sin', 'hour_cos', 'minute_sin', 'minute_cos', 'time_sin', 'time_cos', 'morning_rush', 'evening_rush', 'night', 'rush_hour', 'time_period', 'geo_prefix', 'geo_prefix3', 'geo_len', 'temp_missing', 'road_missing']


In [4]:
# ── 4. Target Encoding for Geohash ────────────────────────────────────────────
# Mean demand per geohash (with smoothing to avoid overfitting)
global_mean = train['demand'].mean()

def target_encode(train_df, test_df, col, target='demand', smoothing=20):
    """Target encoding with smoothing."""
    stats = train_df.groupby(col)[target].agg(['mean', 'count'])
    stats['smoothed'] = (stats['count'] * stats['mean'] + smoothing * global_mean) / (stats['count'] + smoothing)
    mapping = stats['smoothed'].to_dict()
    
    train_df[f'{col}_target_enc'] = train_df[col].map(mapping).fillna(global_mean)
    test_df[f'{col}_target_enc']  = test_df[col].map(mapping).fillna(global_mean)
    return train_df, test_df

# Target encode geohash and geo_prefix
train, test = target_encode(train, test, 'geohash', smoothing=15)
train, test = target_encode(train, test, 'geo_prefix', smoothing=30)
train, test = target_encode(train, test, 'geo_prefix3', smoothing=50)

print('Target encoding done.')
print(f"Geohash target enc — train sample: {train['geohash_target_enc'].head().values}")

Target encoding done.
Geohash target enc — train sample: [0.05688975 0.19220499 0.12171365 0.03531821 0.04831225]


In [5]:
# ── 5. Encode Categoricals ────────────────────────────────────────────────────
CAT_COLS = ['geohash', 'geo_prefix', 'geo_prefix3', 'RoadType', 'LargeVehicles',
            'Landmarks', 'Weather', 'time_period']

combined = pd.concat([train, test], axis=0, ignore_index=True)

encoders = {}
for col in CAT_COLS:
    if col in combined.columns:
        le = LabelEncoder()
        combined[col] = le.fit_transform(combined[col].astype(str))
        encoders[col] = le

train = combined.iloc[:len(train)].copy()
test  = combined.iloc[len(train):].copy()
print('Label encoding done. Shape:', train.shape)

Label encoding done. Shape: (77299, 33)


In [6]:
# ── 6. Define Features & Target ───────────────────────────────────────────────
DROP_COLS = ['Index', 'demand', 'timestamp']  # Keep 'day'!
FEATURE_COLS = [c for c in train.columns if c not in DROP_COLS]

X      = train[FEATURE_COLS]
y      = train['demand']
X_test = test[FEATURE_COLS]

print(f'Features ({len(FEATURE_COLS)}):', FEATURE_COLS)
print(f'X shape: {X.shape} | y shape: {y.shape}')

Features (30): ['geohash', 'day', 'RoadType', 'NumberofLanes', 'LargeVehicles', 'Landmarks', 'Temperature', 'Weather', 'hour', 'minute', 'time_minutes', 'hour_sin', 'hour_cos', 'minute_sin', 'minute_cos', 'time_sin', 'time_cos', 'morning_rush', 'evening_rush', 'night', 'rush_hour', 'time_period', 'geo_prefix', 'geo_prefix3', 'geo_len', 'temp_missing', 'road_missing', 'geohash_target_enc', 'geo_prefix_target_enc', 'geo_prefix3_target_enc']
X shape: (77299, 30) | y shape: (77299,)


In [7]:
# ── 7. Handle Missing Values ──────────────────────────────────────────────────
# Use train median for both (avoid leakage)
train_medians = X.median(numeric_only=True)
X      = X.fillna(train_medians)
X_test = X_test.fillna(train_medians)

# Fill any remaining NaN with 0
X      = X.fillna(0)
X_test = X_test.fillna(0)

print('NaN check train:', X.isnull().sum().sum())
print('NaN check test :', X_test.isnull().sum().sum())

NaN check train: 0
NaN check test : 0


In [8]:
# ── 8. Optuna Hyperparameter Tuning for LightGBM ─────────────────────────────
def objective_lgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 500, 3000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'max_depth': trial.suggest_int('max_depth', 5, 15),
        'num_leaves': trial.suggest_int('num_leaves', 31, 255),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'random_state': 42,
        'n_jobs': -1,
        'verbose': -1,
    }
    model = lgb.LGBMRegressor(**params)
    scores = cross_val_score(model, X, y, cv=5, scoring='r2', n_jobs=-1)
    return scores.mean()

print('Starting Optuna tuning for LightGBM...')
study_lgb = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study_lgb.optimize(objective_lgb, n_trials=50, show_progress_bar=True)

best_lgb_params = study_lgb.best_params
print(f'\nBest LightGBM CV R²: {study_lgb.best_value:.6f}')
print(f'Best params: {best_lgb_params}')

Starting Optuna tuning for LightGBM...


  0%|          | 0/50 [00:00<?, ?it/s]


Best LightGBM CV R²: 0.867644
Best params: {'n_estimators': 1214, 'learning_rate': 0.01694141274541558, 'max_depth': 9, 'num_leaves': 181, 'min_child_samples': 33, 'subsample': 0.823007060629123, 'colsample_bytree': 0.964586796260085, 'reg_alpha': 1.6902749588425184e-08, 'reg_lambda': 0.000242424550815336}


In [9]:
# ── 9. Optuna Tuning for XGBoost ──────────────────────────────────────────────
def objective_xgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 500, 3000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'max_depth': trial.suggest_int('max_depth', 4, 12),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 20),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'gamma': trial.suggest_float('gamma', 1e-8, 5.0, log=True),
        'random_state': 42,
        'n_jobs': -1,
        'verbosity': 0,
    }
    model = xgb.XGBRegressor(**params)
    scores = cross_val_score(model, X, y, cv=5, scoring='r2', n_jobs=-1)
    return scores.mean()

print('Starting Optuna tuning for XGBoost...')
study_xgb = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study_xgb.optimize(objective_xgb, n_trials=50, show_progress_bar=True)

best_xgb_params = study_xgb.best_params
print(f'\nBest XGBoost CV R²: {study_xgb.best_value:.6f}')
print(f'Best params: {best_xgb_params}')

Starting Optuna tuning for XGBoost...


  0%|          | 0/50 [00:00<?, ?it/s]


Best XGBoost CV R²: 0.863912
Best params: {'n_estimators': 685, 'learning_rate': 0.07741682708742595, 'max_depth': 7, 'min_child_weight': 3, 'subsample': 0.9878918277188711, 'colsample_bytree': 0.9939419178570889, 'reg_alpha': 0.0006050457497707735, 'reg_lambda': 9.645416888584492e-08, 'gamma': 0.003783414954097089}


In [11]:
# ── 10. Train All Models on Full Data ─────────────────────────────────────────
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# --- LightGBM ---
lgb_params = {
    **best_lgb_params,
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1,
}
lgb_model = lgb.LGBMRegressor(**lgb_params)
lgb_cv = cross_val_score(lgb_model, X, y, cv=kf, scoring='r2', n_jobs=-1)
print(f'LightGBM  CV R²: {lgb_cv.round(4)} | Mean: {lgb_cv.mean():.6f} | Score: {max(0,100*lgb_cv.mean()):.2f}')
lgb_model.fit(X, y)

# --- XGBoost ---
xgb_params = {
    **best_xgb_params,
    'random_state': 42,
    'n_jobs': -1,
    'verbosity': 0,
}
xgb_model = xgb.XGBRegressor(**xgb_params)
xgb_cv = cross_val_score(xgb_model, X, y, cv=kf, scoring='r2', n_jobs=-1)
print(f'XGBoost   CV R²: {xgb_cv.round(4)} | Mean: {xgb_cv.mean():.6f} | Score: {max(0,100*xgb_cv.mean()):.2f}')
xgb_model.fit(X, y)

# --- RandomForest (improved) ---
rf_model = RandomForestRegressor(
    n_estimators=500,
    max_depth=None,
    min_samples_leaf=1,
    min_samples_split=3,
    n_jobs=-1,
    random_state=42
)
rf_cv = cross_val_score(rf_model, X, y, cv=kf, scoring='r2', n_jobs=-1)
print(f'RF        CV R²: {rf_cv.round(4)} | Mean: {rf_cv.mean():.6f} | Score: {max(0,100*rf_cv.mean()):.2f}')
rf_model.fit(X, y)

LightGBM  CV R²: [0.9481 0.9492 0.9487 0.9453 0.9484] | Mean: 0.947917 | Score: 94.79
XGBoost   CV R²: [0.9396 0.9443 0.9403 0.941  0.9394] | Mean: 0.940936 | Score: 94.09
RF        CV R²: [0.9528 0.9544 0.9541 0.95   0.9525] | Mean: 0.952751 | Score: 95.28


RandomForestRegressor(min_samples_split=3, n_estimators=500, n_jobs=-1,
                      random_state=42)

In [12]:
# ── 11. Ensemble & Predict ────────────────────────────────────────────────────
# Predict with each model
pred_lgb = lgb_model.predict(X_test)
pred_xgb = xgb_model.predict(X_test)
pred_rf  = rf_model.predict(X_test)

# Weighted ensemble (weight by CV performance)
w_lgb = lgb_cv.mean()
w_xgb = xgb_cv.mean()
w_rf  = rf_cv.mean()
w_total = w_lgb + w_xgb + w_rf

y_pred = (w_lgb * pred_lgb + w_xgb * pred_xgb + w_rf * pred_rf) / w_total

print(f'Ensemble weights — LGB: {w_lgb/w_total:.3f}, XGB: {w_xgb/w_total:.3f}, RF: {w_rf/w_total:.3f}')
print(f'Predictions range: [{y_pred.min():.4f}, {y_pred.max():.4f}]')
print(f'Predictions mean:  {y_pred.mean():.4f}')

Ensemble weights — LGB: 0.334, XGB: 0.331, RF: 0.335
Predictions range: [0.0054, 1.0419]
Predictions mean:  0.1269


In [13]:
# ── 12. Save Submission ───────────────────────────────────────────────────────
submission = pd.DataFrame({
    'Index' : test['Index'].values,
    'demand': y_pred
})

submission.to_csv('submission_v2.csv', index=False)
print('Saved submission_v2.csv')
display(submission.head(10))
print(f'Shape: {submission.shape}')  # should be 41778 × 2

Saved submission_v2.csv


,Index,demand
0,0,0.072099
1,1,0.027341
2,2,0.052710
3,3,0.041019
4,4,0.054023
5,5,0.044289
6,6,0.037369
7,7,0.070139
8,8,0.032762
9,9,0.050225


Shape: (41778, 2)


In [14]:
# ── 13. Feature Importance (LightGBM) ─────────────────────────────────────────
importance = pd.Series(lgb_model.feature_importances_, index=FEATURE_COLS)
importance = importance.sort_values(ascending=False)
print('Top 15 features (LightGBM):')
display(importance.head(15))

# Also show XGBoost importance
xgb_imp = pd.Series(xgb_model.feature_importances_, index=FEATURE_COLS)
xgb_imp = xgb_imp.sort_values(ascending=False)
print('\nTop 15 features (XGBoost):')
display(xgb_imp.head(15))

Top 15 features (LightGBM):


geohash                  36907
geohash_target_enc       35827
Temperature              15805
time_minutes             12483
time_cos                  7506
time_sin                  7343
hour                      3866
RoadType                  3323
NumberofLanes             3138
hour_sin                  3086
day                       2285
hour_cos                  2162
minute                    2120
Weather                   1838
geo_prefix_target_enc     1755
dtype: int32


Top 15 features (XGBoost):


RoadType                 0.758120
LargeVehicles            0.106981
geohash_target_enc       0.036173
hour_sin                 0.024250
hour_cos                 0.012558
evening_rush             0.009655
time_cos                 0.006819
NumberofLanes            0.006729
Landmarks                0.006540
time_minutes             0.004308
day                      0.004223
hour                     0.004092
time_sin                 0.003608
geohash                  0.002987
geo_prefix_target_enc    0.002254
dtype: float32

In [15]:
# ── 14. Try K-Fold Ensemble (more robust) ─────────────────────────────────────
# Train 5 folds and average predictions for even better generalization
oof_lgb = np.zeros(len(X))
oof_xgb = np.zeros(len(X))
test_lgb = np.zeros(len(X_test))
test_xgb = np.zeros(len(X_test))

for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    # LightGBM fold
    m_lgb = lgb.LGBMRegressor(**lgb_params)
    m_lgb.fit(X_tr, y_tr)
    oof_lgb[val_idx] = m_lgb.predict(X_val)
    test_lgb += m_lgb.predict(X_test) / 5
    
    # XGBoost fold
    m_xgb = xgb.XGBRegressor(**xgb_params)
    m_xgb.fit(X_tr, y_tr)
    oof_xgb[val_idx] = m_xgb.predict(X_val)
    test_xgb += m_xgb.predict(X_test) / 5
    
    fold_r2_lgb = metrics.r2_score(y_val, oof_lgb[val_idx])
    fold_r2_xgb = metrics.r2_score(y_val, oof_xgb[val_idx])
    print(f'Fold {fold+1}: LGB R²={fold_r2_lgb:.6f} | XGB R²={fold_r2_xgb:.6f}')

# OOF ensemble
oof_ensemble = (w_lgb * oof_lgb + w_xgb * oof_xgb) / (w_lgb + w_xgb)
oof_r2 = metrics.r2_score(y, oof_ensemble)
print(f'\nOOF Ensemble R²: {oof_r2:.6f} | Score: {max(0,100*oof_r2):.2f}')

# Final K-Fold ensemble prediction
y_pred_kfold = (w_lgb * test_lgb + w_xgb * test_xgb) / (w_lgb + w_xgb)

# Save K-Fold submission
submission_kf = pd.DataFrame({
    'Index' : test['Index'].values,
    'demand': y_pred_kfold
})
submission_kf.to_csv('submission_v2_kfold.csv', index=False)
print('Saved submission_v2_kfold.csv')

Fold 1: LGB R²=0.948069 | XGB R²=0.939848
Fold 2: LGB R²=0.949166 | XGB R²=0.944430
Fold 3: LGB R²=0.948668 | XGB R²=0.940436
Fold 4: LGB R²=0.945268 | XGB R²=0.940783
Fold 5: LGB R²=0.948413 | XGB R²=0.938917

OOF Ensemble R²: 0.946373 | Score: 94.64
Saved submission_v2_kfold.csv


In [16]:
# ── 15. Summary ───────────────────────────────────────────────────────────────
print('='*60)
print('SUMMARY')
print('='*60)
print(f'LightGBM  CV R²: {lgb_cv.mean():.6f} → Score: {max(0,100*lgb_cv.mean()):.2f}')
print(f'XGBoost   CV R²: {xgb_cv.mean():.6f} → Score: {max(0,100*xgb_cv.mean()):.2f}')
print(f'RF        CV R²: {rf_cv.mean():.6f} → Score: {max(0,100*rf_cv.mean()):.2f}')
print(f'OOF Ensemble R²: {oof_r2:.6f} → Score: {max(0,100*oof_r2):.2f}')
print('='*60)
print('Submissions saved:')
print('  1. submission_v2.csv       — weighted ensemble (full-data trained)')
print('  2. submission_v2_kfold.csv — K-Fold ensemble (more robust)')
print('Try both and submit the better one!')

SUMMARY
LightGBM  CV R²: 0.947917 → Score: 94.79
XGBoost   CV R²: 0.940936 → Score: 94.09
RF        CV R²: 0.952751 → Score: 95.28
OOF Ensemble R²: 0.946373 → Score: 94.64
Submissions saved:
  1. submission_v2.csv       — weighted ensemble (full-data trained)
  2. submission_v2_kfold.csv — K-Fold ensemble (more robust)
Try both and submit the better one!
